# Model Comparison: Logistic Regression · Random Forest · KNN
**Dataset:** CIC-IDS2017 · DoS Flows

This notebook loads and cleans the dataset, then trains three classifiers independently
and prints a clean results summary for each.

## 1. Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score,ConfusionMatrixDisplay, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier


## 2. Load Dataset

In [4]:
files = sorted(glob("combined_part_*.csv"))

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f"Raw shape: {df.shape}")


ValueError: No objects to concatenate

## 3. Data Cleaning

In [ ]:
# Remove Trailing whitespaces
df.columns = df.columns.str.strip()

# Remove duplicates and inf numbers
df = df.drop_duplicates()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Remove Heartbleed (low number of samples)
df = df[df["Label"] != "Heartbleed"]

print("Class Distribution:")
print(df["Label"].value_counts())

## 4. Feature Selection & Train/Test Split

In [ ]:
TOP_FEATURES = [
    "Avg Bwd Segment Size", "Max Packet Length", "Bwd Packet Length Max",
    "Packet Length Std", "Packet Length Mean", "Bwd Packet Length Mean",
    "Average Packet Size", "Packet Length Variance", "Subflow Bwd Bytes",
    "Total Length of Bwd Packets", "Fwd Packet Length Max", "Bwd Packet Length Std",
    "Init_Win_bytes_forward", "Flow Duration", "Flow Bytes/s", "Flow IAT Mean",
    "Fwd IAT Total", "Flow IAT Std", "Idle Mean", "Active Mean",
]

X = df[TOP_FEATURES]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Features: {len(TOP_FEATURES)}")


## 5. Evaluation Helper

Shared function used by all three models to print results consistently.

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    print("="*60)
    print(f"  {name}")
    print("="*60)

    # ── Fit ──────────────────────────────────────────────────────
    model.fit(X_tr, y_tr)

    # ── Predictions ──────────────────────────────────────────────
    train_pred = model.predict(X_tr)
    test_pred  = model.predict(X_te)

    train_acc  = accuracy_score(y_tr, train_pred)
    test_acc   = accuracy_score(y_te, test_pred)
    test_f1    = f1_score(y_te, test_pred, average="weighted")

    # ── Summary metrics ──────────────────────────────────────────
    print(f"  Train Accuracy : {train_acc:.4f}")
    print(f"  Test  Accuracy : {test_acc:.4f}")
    print(f"  Test  F1 (weighted): {test_f1:.4f}")
    gap = train_acc - test_acc
    print(f"  Overfitting Gap: {gap:+.4f}  ", end="")
    if abs(gap) < 0.01:
        print("(no overfitting)")
    elif abs(gap) < 0.05:
        print("(slight overfitting)")
    else:
        print("(overfitting)")

    # ── Classification report ────────────────────────────────────
    print()
    print(classification_report(y_te, test_pred))

   
results = []

## 6. Model 1 – Logistic Regression

Linear baseline. Requires feature scaling, therefore wrapped in a Pipeline.

In [ ]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ))
])

lr_results = evaluate_model(
    "Logistic Regression",
    lr_pipeline,
    X_train, y_train,
    X_test,  y_test,
)
results.append(lr_results)


## 7. Model 2 – Random Forest

Ensemble of decision trees. No scaling needed.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

rf_results = evaluate_model(
    "Random Forest",
    rf_model,
    X_train, y_train,
    X_test,  y_test,
)
results.append(rf_results)


### Feature Importance (Random Forest)

In [ ]:
importance = pd.Series(
    rf_model.feature_importances_,
    index=TOP_FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Random Forest – Feature Importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()


## 8. Model 3 – K-Nearest Neighbors (KNN)

Distance-based classifier. Requires feature scaling.

In [ ]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        n_jobs=-1,
    ))
])

knn_results = evaluate_model(
    "KNN (k=5, distance-weighted)",
    knn_pipeline,
    X_train, y_train,
    X_test,  y_test,
)
results.append(knn_results)


## 9. Results Summary

In [ ]:
summary = pd.DataFrame(results).set_index("Model")

# Highlight best value per column
def highlight_best(s):
    if s.name in ["Train Acc", "Test Acc", "F1 (weighted)"]:
        best = s.max()
        return ["font-weight: bold; color: green" if v == best else "" for v in s]
    else:  # Overfit Gap: closest to 0 is best
        best = s.abs().min()
        return ["font-weight: bold; color: green" if abs(v) == best else "" for v in s]

display(summary.style.apply(highlight_best).format({
    "Train Acc":     "{:.4f}",
    "Test Acc":      "{:.4f}",
    "F1 (weighted)": "{:.4f}",
    "Overfit Gap":   "{:+.4f}",
}))


### Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

metrics = ["Train Acc", "Test Acc", "F1 (weighted)"]
colors  = ["#4C72B0", "#DD8452", "#55A868"]

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.bar(summary.index, summary[metric], color=color, width=0.5)
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=15)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f"{h:.4f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Model Comparison – CIC-IDS2017 DoS Dataset", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
